# ALAE latent space — CHILDREN

## Imports

In [ ]:
import sys

sys.path.append("..")
sys.path.append("../ALAE")

import math

import numpy as np
import torch
from matplotlib import pyplot as plt
from sklearn.cluster import KMeans
from tqdm.auto import tqdm

from src.schr_bridge import Schr_Bridge
from src.distributions import TensorSampler, TruncatedNormalSampler

from alae_ffhq_inference import load_model, decode

%matplotlib inline

## Configuration

In [ ]:
DIM = 512
TARGET_DATA = "CHILDREN"

OUTPUT_SEED = 42
BATCH_SIZE = 128
D_LR = 1e-4
D_GRADIENT_MAX_NORM = float("inf")
MAX_STEPS = 10000

EPSILON = 0.2
N_POTENTIALS = 50
B = 0.917004
M = 0.0
T = 15.0
S_DIAGONAL_INIT = 0.5
S_LOWER_BOUND = 0.05
IS_DIAGONAL = True
SAMPLING_BATCH_SIZE = 128
INIT_BY_SAMPLES = True

M_SLACK = 3.0
CONSTRAINT_WEIGHT = 5.0

GALLERY_PREFIX = "childs"
GRID_ROWS, GRID_COLS = 5, 5

TRAIN_SIZE = 60000
DOWNLOAD_DATA = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Data

`../data/latents.npy` holds the ALAE encodings of the 70000 FFHQ images
(70000 x 512), `../data/gender.npy` and `../data/age.npy` the attributes of the
same images; the first `TRAIN_SIZE` rows form the training split. Set
`DOWNLOAD_DATA = True` in the configuration cell above and run the next cell
once to download the three files (~150 MB) from Google Drive.

In [ ]:
if DOWNLOAD_DATA:
    import gdown

    urls = {
        "../data/age.npy": "https://drive.google.com/uc?id=1Vi6NzxCsS23GBNq48E-97Z9UuIuNaxPJ",
        "../data/gender.npy": "https://drive.google.com/uc?id=1SEdsmQGL3mOok1CPTBEfc_O1750fGRtf",
        "../data/latents.npy": "https://drive.google.com/uc?id=1ENhiTRsHtSjIjoRu1xYprcpNd8M9aVu8",
    }
    for path, url in urls.items():
        gdown.download(url, path, quiet=False)

In [ ]:
latents = np.load("../data/latents.npy")
gender = np.load("../data/gender.npy").reshape(-1)
age = np.load("../data/age.npy").reshape(-1)

train_latents, train_gender, train_age = (
    latents[:TRAIN_SIZE],
    gender[:TRAIN_SIZE],
    age[:TRAIN_SIZE],
)

if TARGET_DATA == "MAN":
    target_mask = train_gender == "male"
elif TARGET_DATA == "WOMAN":
    target_mask = train_gender == "female"
elif TARGET_DATA == "ADULT":
    target_mask = (train_age >= 18) & (train_age != -1)
elif TARGET_DATA == "CHILDREN":
    target_mask = (train_age < 18) & (train_age != -1)
else:
    raise ValueError(f"unknown TARGET_DATA: {TARGET_DATA}")

Y_train = torch.tensor(train_latents[target_mask])
print(f"{TARGET_DATA}: {Y_train.shape[0]} target latents of dimension {Y_train.shape[1]}")

## Model

In [ ]:
def compute_loss(model, X1, t):
    log_potential = model.get_log_potential(X1)
    log_C_tilde = model.get_log_C_tilde(t)

    b = model.b
    e_sq = math.e ** 2
    K_T = ((1 - torch.exp(-2 * b * t)) ** (-5 * e_sq * math.sqrt(model.dim))
           * torch.exp(2 * e_sq * math.sqrt(model.dim) * torch.arcsin(torch.exp(-b * t))))

    sigma_T_sq = torch.clamp((1 - torch.exp(-2 * b * t)) / (2 * b), min=1e-8)

    quad = (X1 ** 2).sum(dim=-1) / (2 * model.epsilon * sigma_T_sq)
    phi = log_potential + quad
    penalty = torch.relu(phi - (quad.max() + M_SLACK)).mean()

    return ((1.0 / K_T) * log_C_tilde - log_potential).mean() + CONSTRAINT_WEIGHT * penalty


def kmeans_init_r(Y, K, seed=OUTPUT_SEED, max_pts=5000):
    Yc = torch.as_tensor(Y, dtype=torch.float32).reshape(-1, Y.shape[-1]).cpu().numpy()
    if Yc.shape[0] > max_pts:
        idx = np.random.RandomState(seed).choice(Yc.shape[0], max_pts, replace=False)
        Yc = Yc[idx]
    centers = KMeans(n_clusters=K, random_state=seed, n_init=4).fit(Yc).cluster_centers_
    return torch.tensor(centers, dtype=torch.float32)

## Training

In [ ]:
torch.manual_seed(OUTPUT_SEED)
np.random.seed(OUTPUT_SEED)

X_sampler = TruncatedNormalSampler(DIM, device=DEVICE)
Y_sampler = TensorSampler(Y_train, device=DEVICE)

D = Schr_Bridge(
    dim=DIM,
    n_potentials=N_POTENTIALS,
    epsilon=EPSILON,
    b=B,
    m=M,
    T=T,
    sampling_batch_size=SAMPLING_BATCH_SIZE,
    S_diagonal_init=S_DIAGONAL_INIT,
    S_lower_bound=S_LOWER_BOUND,
    is_diagonal=IS_DIAGONAL,
).to(DEVICE)

if INIT_BY_SAMPLES:
    D.init_r_by_samples(kmeans_init_r(Y_train, N_POTENTIALS))

D_opt = torch.optim.Adam(D.parameters(), lr=D_LR)

loss_history, grad_norm_history = [], []
for step in tqdm(range(MAX_STEPS)):
    D_opt.zero_grad()

    X1 = Y_sampler.sample(BATCH_SIZE)
    D_loss = compute_loss(D, X1, t=D.T)
    D_loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=D_GRADIENT_MAX_NORM)
    D_opt.step()

    loss_history.append(D_loss.item())
    grad_norm_history.append(grad_norm.item())

print(f"mean loss over the last 100 steps: {np.mean(loss_history[-100:]):.4f}")

## Decoder

Decoding needs the pretrained ALAE FFHQ generator in
`../ALAE/training_artifacts/ffhq/`. The script below downloads
`model_submitted.pth`, `model_194.pth` and `model_157.pth` (~1.3 GB in total)
from the Google Drive of the ALAE authors, falling back to their S3 bucket, and
the second line selects the checkpoint to load:

```
cd ../ALAE
python training_artifacts/download_all.py
echo training_artifacts/ffhq/model_157.pth > training_artifacts/ffhq/last_checkpoint
```

In [ ]:
model = load_model(
    "../ALAE/configs/ffhq.yaml",
    training_artifacts_dir="../ALAE/training_artifacts/ffhq/",
).to(DEVICE).eval()

## Visualization

In [ ]:
torch.manual_seed(OUTPUT_SEED)
np.random.seed(OUTPUT_SEED)


def decode_to_imgs(latents):
    with torch.no_grad(), torch.device(DEVICE):
        img = decode(model, latents)
        denom = (img.max() - img.min()) / 2
        img = ((img / denom + 0.5) * 255).type(torch.long).clamp(0, 255)
        return img.cpu().type(torch.uint8).permute(0, 2, 3, 1).numpy()


n_gallery = GRID_ROWS * GRID_COLS
with torch.no_grad(), torch.device(DEVICE):
    gen_latents = D(X_sampler.sample(n_gallery))
gen_imgs = decode_to_imgs(gen_latents)

from PIL import Image

gallery = Image.new("RGB", (GRID_COLS * 256, GRID_ROWS * 256))
for i, pixels in enumerate(gen_imgs):
    tile = Image.fromarray(pixels).resize((256, 256), Image.Resampling.LANCZOS)
    gallery.paste(tile, ((i % GRID_COLS) * 256, (i // GRID_COLS) * 256))
gallery.save(f"{GALLERY_PREFIX}_generation_gallery.jpg", quality=95)
display(gallery)